# 06 — Contrôle qualité du catalogue plantes
Ce notebook **ne valide pas automatiquement** les plantes. Il montre la couverture horticole, les doublons et les images documentées. Aucun téléchargement de photo sans droits vérifiés.


In [ ]:
from pathlib import Path
import os,subprocess,sys
from dotenv import load_dotenv
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"database/scripts/plant_quality_report.py").exists())
load_dotenv(ROOT/".env",override=False)
load_dotenv(ROOT/".env.local",override=False)
subprocess.run([sys.executable,str(ROOT/"database/scripts/plant_quality_report.py")],cwd=ROOT,check=True)



In [ ]:
import psycopg,pandas as pd
dsn=os.environ["GARDNAIZE_ML_DATABASE_URL"].replace("postgresql+psycopg://","postgresql://",1)
with psycopg.connect(dsn) as c:
 with c.cursor() as cur:
  cur.execute("""SELECT plant_id,scientific_name,quality_status,
    (exposure IS NOT NULL)::int+(soil_texture IS NOT NULL)::int+
    (water_requirement IS NOT NULL)::int+(hardiness_min_c IS NOT NULL)::int+
    (height_cm IS NOT NULL)::int AS core_fields_present
    FROM horticulture.ml_plant_catalog ORDER BY core_fields_present,plant_id LIMIT 100""")
  display(pd.DataFrame(cur.fetchall(),columns=[d.name for d in cur.description]))

